In [ ]:
from PIL import Image
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

from transformers import CLIPModel, CLIPProcessor

import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torchvision.transforms.v2 as v2 
import torchmetrics as tm

In [2]:
root_path = "/home/stefan/ioai-prep/kits/cuvinte"

device = "cuda"

# Data

In [ ]:
class WordsDataset(Dataset):
    def __init__(self, is_train=True):
        super().__init__()

        self.is_train = is_train
        self.df = pd.read_csv(f"{root_path}/{'train' if is_train else 'test'}.csv")

        self.transforms = v2.Compose(
            [
                v2.Resize([224, 224]),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
            ]
        )

        if 'label' in self.df:
            self.le = LabelEncoder()
            self.le.fit(self.df['label'])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(f"{root_path}/{row['image_path']}")
        img = self.transforms(img)

        if 'label' not in row:
            return img

        lbl = self.le.transform([row['label']])
        lbl = torch.tensor(lbl)
        return img, lbl

    def __len__(self):
        return len(self.df)

In [45]:
full_train_ds = WordsDataset(is_train=True)
test_ds = WordsDataset(is_train=False)

train_ds, val_ds = torch.utils.data.random_split(full_train_ds, [0.8, 0.2])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

In [46]:
# sanity check
x = next(iter(train_loader))
[b.shape for b in x]

[torch.Size([16, 3, 224, 224]), torch.Size([16, 1])]

# Model

In [47]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)

In [48]:
class WordExtractorModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.vit = clip.vision_model

        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.LazyLinear(64),
            nn.Dropout(0.4),
            nn.ReLU(),
            nn.Linear(64, 20),
        )

    def forward(self, x):
        processor.image_processor(x, return_tensors="pt", do_rescale=False)[
            "pixel_values"
        ].to(device)

        logits = self.vit(x).pooler_output
        out = self.head(logits)
        return out

In [49]:
model = WordExtractorModel().to(device)

model(x[0].to(device)).shape

torch.Size([16, 20])

# Training

In [50]:
lr = 1e-5
epochs = 5

optim = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
accuracy = tm.Accuracy(task="multiclass", num_classes=20).to(device)

In [51]:
for epoch in range(1, epochs+1):
    running_loss = 0
    accuracy.reset()

    for img, label in tqdm(train_loader):
        img, label = img.to(device), label.squeeze(1).to(device)

        logits = model(img)
        loss = criterion(logits, label)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item()

    for img, label in tqdm(val_loader):
        img, label = img.to(device), label.squeeze(1).to(device)

        logits = model(img)

        accuracy(logits, label)

    running_loss /= len(train_loader)
    print(f"epoch {epoch}, loss={running_loss:.3f}, acc={accuracy.compute():.2f}")

100%|██████████| 5/5 [00:02<00:00,  2.08it/s]


epoch 1, loss=2.855, acc=0.26


100%|██████████| 5/5 [00:02<00:00,  2.04it/s]


epoch 2, loss=1.977, acc=0.75


100%|██████████| 5/5 [00:02<00:00,  2.03it/s]


epoch 3, loss=1.147, acc=0.94


100%|██████████| 5/5 [00:02<00:00,  2.02it/s]


epoch 4, loss=0.620, acc=1.00


100%|██████████| 5/5 [00:02<00:00,  1.97it/s]

epoch 5, loss=0.406, acc=1.00


# Submission

In [52]:
answers = []

for img in tqdm(test_loader):
    logits = model(img.to(device))
    answer = torch.argmax(logits, dim=1)
    answers.extend(answer.cpu().tolist())

100%|██████████| 7/7 [00:04<00:00,  1.55it/s]


In [53]:
answers = full_train_ds.le.inverse_transform(answers)

In [54]:
sub = pd.read_csv(f"{root_path}/test.csv")
sub["label"] = answers
sub.head()

,image_path,label
0,output_dataset/test/000000.png,tree
1,output_dataset/test/000001.png,dog
2,output_dataset/test/000002.png,sun
3,output_dataset/test/000003.png,elephant
4,output_dataset/test/000004.png,tree


In [55]:
sub.to_csv(f"{root_path}/submission.csv", index=False)